In [ ]:
import torch

print('GPU Available:', torch.cuda.is_available())
print('Device Name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
import sys
sys.path.append(r"g:\pii_env")

from app import HybridPIIEngine


sample_text = """
[Call Date: 02/03/2026 | Duration: 22:15 | Agent: David Ramirez | Channel: Phone]
Agent: Thank you for calling United Family Insurance. This is David Ramirez, badge number UFI-33291. How may I assist you?
Customer: Hi David. My name is Dr. Priya Anand Chakraborty. I'm calling about a car accident I was in last week and I need to file claims under both my auto and health insurance.
Agent: I'm sorry to hear that, Dr. Chakraborty. Let me pull up your account. Can you verify your policy number and date of birth?
Customer: My auto policy is UFI-AUTO-5538291. My health insurance is through the same company, policy UFI-HEALTH-5538292. Group number GRP-BIOTECH-4421. My date of birth is November 3, 1985. SSN is 621-73-4489.
Agent: Verified. Can you walk me through what happened?
Customer: On January 28th at about 3:15 PM, I was rear-ended at the intersection of Camelback Road and 24th Street in Phoenix. The other driver's name was Robert James Martinez, and he gave me his license number — it was an Arizona license, D98321754. His insurance is State Farm, policy number SF-229-4817362-01. His plate number was Arizona FGR-8821. His vehicle was a 2019 Ford F-150, VIN 1FTEW1EP5KFA82943.
Agent: And your vehicle information?
Customer: 2024 Audi Q7, VIN WAUZZZ4M1RD019384, Arizona plate NLV-3392. It's leased through Audi Financial Services, account AFS-22918743. My lien holder address is on file.
Agent: Got it. Now for the medical portion — can you describe your injuries and treatment?
Customer: I went to the Scottsdale Osborn Medical Center ER that evening. My medical record number there is SMC-00482917. I was diagnosed with cervical whiplash, a mild concussion — ICD-10 code S13.4XXA and S06.0X0A. They did a CT scan and X-rays. The attending physician was Dr. Leonard Wu, NPI number 1497825310. I was prescribed Cyclobenzaprine 10mg, prescription number RX-2294817, filled at CVS pharmacy on Shea Boulevard, store number 6742.
Agent: Have you had any follow-up treatment?
Customer: Yes. I'm seeing Dr. Amanda Foster for physical therapy, NPI 1832749162, at Desert Spine & Rehabilitation. My patient ID there is DSR-P-08291. I've had three sessions so far. I also saw my primary care physician, Dr. Rajesh Gupta, NPI 1628374910, at Banner Health, patient ID BH-44829173. He referred me to a neurologist for the concussion follow-up. Oh, and my Medicare Beneficiary ID is 1EK4-TA2-HY74 since I also have Medicare Part B through my disability status.
Agent: Thank you. For the auto claim, can I get the police report number?
Customer: Phoenix PD report number 2026-PX-0028471. The responding officer was Officer Kevin Doyle, badge number PPD-4419. There were two witnesses — Maria Santos, phone 602-555-8837, and James Liu, phone 480-555-2291.
Agent: I need a few more details for your health claim. Can you verify your blood type and any existing conditions in your medical history that might be relevant?
Customer: Blood type is O-negative. I have a pre-existing condition of Type 2 diabetes, currently managed with Metformin. I also have a medical device — a continuous glucose monitor, device serial number DX-G6-482917384. My pharmacy benefit manager is Express Scripts, member ID ESI-77482913.
Agent: And your emergency contact on file?
Customer: My husband, Arjun Chakraborty. Cell phone 480-555-6614. He's my authorized representative on all accounts. His SSN is 621-73-4490 and date of birth is June 17, 1983.
Agent: Perfect. I've opened auto claim number UFI-AC-2026-019384 and health claim number UFI-HC-2026-019385. A claims adjuster will contact you within 48 hours. Your deductible on the auto is $500 and the health deductible has been met for the year. Is there anything else?
Customer: Can you send the claim documents to my work email? It's pchakraborty@genedyne-biotech.com. My employee ID there is GBT-E-4817. Actually, also copy my attorney — Patricia Huang, Esq., at phuang@desertlawgroup.com, phone 602-555-9918, bar number AZ-029481.
Agent: Done. Anything else, Dr. Chakraborty?
Customer: No, that covers it. Thank you, David.
"""

engine = HybridPIIEngine(
    use_gliner=True,
    gliner_model_name="knowledgator/gliner-pii-large-v1.0",
    gliner_threshold=0.15,
)

result = engine.redact(sample_text)

print("DETECTIONS:")
for d in result.detections:
    xv = " [XV]" if d.meta.get("cross_validated") else ""
    print(
        f"{d.label:26} {d.text!r:45} "
        f"{d.score:.2f} {d.source:16} "
        f"{d.meta.get('instance_label', '')}{xv}"
    )

print(f"\nTotal: {len(result.detections)} detections")
print(f"Cross-validated: {sum(1 for d in result.detections if d.meta.get('cross_validated'))} detections")
print(f"Unknown candidates: {len(result.unknown_candidates)}")
print(f"Learning stats: {engine.learning_stats()}")

print("\nREDACTED:\n")
print(result.redacted_text)